<a href="https://colab.research.google.com/github/RaheemKProjects/Raheem-North-Star-Analytical-Workflow/blob/main/01_sql_in_r_sqldf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/01_sql_in_r_sqldf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — SQL in R using sqldf

NorthStar Analytics - Raheem

This notebook executes six diagnostic SQL queries against the NorthStar dataset using the `sqldf` package within an R runtime. It corresponds to Section 3 of the report.



## 1. Install and load packages

In [1]:
install.packages(c('sqldf', 'RCurl', 'dplyr', 'ggplot2', 'lubridate'),
                 quiet = TRUE)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’, ‘bitops’




In [2]:
library(sqldf); library(RCurl); library(dplyr); library(ggplot2); library(lubridate)
set.seed(42)
options(stringsAsFactors = FALSE)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




## 2. Load NorthStar datasets

Files are loaded from the local Colab session. To load from GitHub instead, replace each `read.csv` call with `read.csv(text = getURL(...))` using the raw GitHub URL (the `RCurl` bridge to web data taught in Week 4).

In [3]:
# Upload the CSVs via Colab's Files panel, or mount Drive.
# All files should be in the working directory.

customers  <- read.csv('customers.csv')
orders     <- read.csv('orders.csv')
deliveries <- read.csv('deliveries.csv')
drivers    <- read.csv('drivers.csv')
vehicles   <- read.csv('vehicles.csv')
hubs       <- read.csv('hubs.csv')
incidents  <- read.csv('incidents.csv')
complaints <- read.csv('complaints.csv')

cat('Rows loaded:\n')
cat('  customers :', nrow(customers),  '\n')
cat('  orders    :', nrow(orders),     '\n')
cat('  deliveries:', nrow(deliveries), '\n')
cat('  drivers   :', nrow(drivers),    '\n')
cat('  vehicles  :', nrow(vehicles),   '\n')
cat('  hubs      :', nrow(hubs),       '\n')
cat('  incidents :', nrow(incidents),  '\n')
cat('  complaints:', nrow(complaints), '\n')

Rows loaded:
  customers : 650 
  orders    : 1250 
  deliveries: 950 
  drivers   : 170 
  vehicles  : 120 
  hubs      : 8 
  incidents : 280 
  complaints: 320 


## 3. Data cleaning — zone normalisation

The pickup_zone, dropoff_zone, home_zone and base_zone fields contain 16 distinct labels representing only 8 logical zones. Without this step every zone-level aggregation is fragmented and misleading.

In [4]:
normalise_zone <- function(x) {
  x <- toupper(trimws(x))
  x[x %in% c('CTR','CENTRAL')]   <- 'Central'
  x[x == 'AIRPORT']              <- 'Airport'
  x[x == 'NORTH']                <- 'North'
  x[x == 'SOUTH']                <- 'South'
  x[x == 'EAST']                 <- 'East'
  x[x == 'WEST']                 <- 'West'
  x[x == 'RIVERSIDE']            <- 'Riverside'
  return(x)
}

orders$pickup_zone   <- normalise_zone(orders$pickup_zone)
orders$dropoff_zone  <- normalise_zone(orders$dropoff_zone)
customers$home_zone  <- normalise_zone(customers$home_zone)
drivers$base_zone    <- normalise_zone(drivers$base_zone)

cat('Distinct zones in orders$pickup_zone after normalisation:\n')
print(sort(unique(orders$pickup_zone)))

Distinct zones in orders$pickup_zone after normalisation:
[1] "Airport"   "Central"   "East"      "North"     "Riverside" "South"    
[7] "West"     


## 4. Query 1 — Zone-level delivery performance (multi-table JOIN)

In [5]:
zone_perf <- sqldf("
  SELECT
    o.pickup_zone                                                  AS zone,
    COUNT(d.delivery_id)                                           AS n_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed'
              THEN 1 ELSE 0 END) / COUNT(d.delivery_id), 2)        AS failure_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 3)                 AS avg_rating,
    ROUND(AVG(d.manual_route_override_count), 3)                   AS avg_overrides
  FROM orders o
  INNER JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")
print(zone_perf)

       zone n_deliveries failed delayed failure_rate_pct avg_rating
1   Central          174     33      51            18.97      3.546
2     North          135     22      21            16.30      3.897
3 Riverside          119     18      25            15.13      3.864
4      West          114     14      21            12.28      3.896
5      East          156     19      31            12.18      3.912
6   Airport          113     12      31            10.62      3.984
7     South          139     14      22            10.07      4.052
  avg_overrides
1         1.293
2         0.696
3         0.731
4         0.807
5         0.788
6         1.805
7         0.691


**Interpretation:** Central zone has the highest failure rate, more than three times that of South. Optimisation note: conditional aggregation with SUM(CASE WHEN ...) computes all status counts in a single pass.

## 5. Query 2 — The smoking gun: vehicle maintenance status (LEFT JOIN)

In [6]:
maintenance_outcome <- sqldf("
  SELECT v.maintenance_status,
         COUNT(d.delivery_id)                              AS n_dispatches,
         SUM(CASE WHEN d.delivery_status='Failed'  THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN d.delivery_status='Delayed' THEN 1 ELSE 0 END) AS delayed,
         ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END) / COUNT(d.delivery_id), 2)      AS failure_pct,
         ROUND(AVG(v.battery_health_pct), 2)               AS avg_battery_pct
  FROM vehicles v
  LEFT JOIN deliveries d ON v.vehicle_id = d.vehicle_id
  GROUP BY v.maintenance_status
  ORDER BY failure_pct DESC
")
print(maintenance_outcome)

  maintenance_status n_dispatches failed delayed failure_pct avg_battery_pct
1           InRepair          254     77      52       30.31           76.73
2             Active          542     45     113        8.30           76.56
3          Scheduled          154     10      37        6.49           78.74


**Interpretation:** InRepair vehicles fail at 30.31% — more than 3.6 times the rate of Active vehicles (8.31%). Battery health is statistically identical across the three states, indicating the InRepair flag is set reactively after incidents rather than predictively. This single finding accounts for 58.3% of all failures in the dataset.

## 6. Query 3 — Hub-level diagnostics (three-table JOIN)

In [7]:
hub_diagnostics <- sqldf("
  SELECT h.hub_id, h.hub_name, h.hub_type, h.zone, h.capacity_score,
         COUNT(d.delivery_id) AS volume,
         ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END) / COUNT(d.delivery_id), 2) AS failure_pct,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM hubs h
  INNER JOIN deliveries d ON h.hub_id = d.hub_id
  INNER JOIN orders     o ON d.order_id = o.order_id
  GROUP BY h.hub_id, h.hub_name, h.hub_type, h.zone, h.capacity_score
  ORDER BY failure_pct DESC
")
print(hub_diagnostics)

  hub_id       hub_name  hub_type      zone capacity_score volume failure_pct
1    H08  Midtown Relay  Charging   Central             63    128       20.31
2    H05   Central Core   Control   Central             88    115       20.00
3    H06    Airport Hub  Dispatch   Airport             71    104       14.42
4    H04      West Gate  Dispatch      West             69    127       12.60
5    H01 North Exchange  Dispatch     North             82    136       12.50
6    H07  Riverside Hub Warehouse Riverside             66    115       12.17
7    H02     South Link  Dispatch     South             78    106        9.43
8    H03      East Dock Warehouse      East             74    119        9.24
  avg_overrides avg_rating
1          1.11       3.88
2          0.95       3.67
3          0.91       3.88
4          0.87       3.92
5          1.03       3.84
6          1.05       3.88
7          0.92       3.95
8          0.89       3.90


## 7. Query 4 — High-risk customers (Subquery with HAVING — optimisation pattern)

In [8]:
high_risk <- sqldf("
  SELECT c.customer_id, c.home_zone, c.customer_type, c.account_status,
         c.loyalty_score, sub.n_complaints, sub.total_compensation,
         sub.avg_resolution_days
  FROM customers c
  INNER JOIN (
    SELECT customer_id,
           COUNT(*)                              AS n_complaints,
           ROUND(SUM(compensation_amount), 2)    AS total_compensation,
           ROUND(AVG(resolution_days), 1)        AS avg_resolution_days
    FROM complaints
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
  ) sub ON c.customer_id = sub.customer_id
  ORDER BY sub.total_compensation DESC
  LIMIT 10
")
print(high_risk)

   customer_id home_zone customer_type account_status loyalty_score
1        C0421   Central      Consumer         Active          59.0
2        C0573   Airport           SME         Active          57.3
3        C0351   Central    Enterprise         Active          48.6
4        C0078      East      Consumer         Active          35.2
5        C0517 Riverside      Consumer         Active          38.5
6        C0368     North      Consumer         Active          49.5
7        C0242      East      Consumer         Active          83.8
8        C0269 Riverside    Enterprise         Active          76.3
9        C0282 Riverside      Consumer         Active          71.4
10       C0545     South      Consumer         Active          66.9
   n_complaints total_compensation avg_resolution_days
1             3             118.98                 9.7
2             3             111.42                10.7
3             2             102.02                12.5
4             2             101.

## 8. Query 5 — Driver performance ranking (Window functions and CTEs)

In [9]:
worst_drivers <- sqldf("
  WITH driver_outcomes AS (
    SELECT dr.driver_id, dr.base_zone, dr.driver_rating, dr.years_experience,
           COUNT(d.delivery_id) AS n_deliveries,
           SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failed,
           ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                    THEN 1 ELSE 0 END) / COUNT(d.delivery_id), 2) AS failure_pct
    FROM drivers dr
    INNER JOIN deliveries d ON dr.driver_id = d.driver_id
    GROUP BY dr.driver_id
    HAVING COUNT(d.delivery_id) >= 3
  ),
  ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY base_zone
                              ORDER BY failure_pct DESC) AS rank_in_zone
    FROM driver_outcomes
  )
  SELECT * FROM ranked WHERE rank_in_zone = 1
  ORDER BY failure_pct DESC
")
print(worst_drivers)

  driver_id base_zone driver_rating years_experience n_deliveries failed
1      D063     North          4.03               12            3      2
2      D092      East          4.24               15            5      3
3      D104      West          3.45               15            7      4
4      D111   Airport          4.12               15            4      2
5      D103   Central          4.40               15            4      2
6      D024 Riverside          3.35                8            8      4
7      D132     South          4.20                8            4      2
  failure_pct rank_in_zone
1       66.67            1
2       60.00            1
3       57.14            1
4       50.00            1
5       50.00            1
6       50.00            1
7       50.00            1


## 9. Query 6 — Service-type profitability (CASE classifier)

In [10]:
service_economics <- sqldf("
  SELECT o.service_type,
         COUNT(o.order_id) AS n_orders,
         ROUND(SUM(o.order_value), 2) AS gross_revenue,
         ROUND(SUM(d.fuel_or_charge_cost), 2) AS direct_cost,
         ROUND(SUM(o.order_value) - SUM(d.fuel_or_charge_cost), 2) AS gross_margin,
         ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END) / COUNT(o.order_id), 2) AS failure_pct,
         CASE
           WHEN 100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END) / COUNT(o.order_id) > 14 THEN 'High Risk'
           WHEN 100.0 * SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END) / COUNT(o.order_id) > 10 THEN 'Medium Risk'
           ELSE 'Low Risk'
         END AS risk_tier
  FROM orders o
  INNER JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.service_type
  ORDER BY failure_pct DESC
")
print(service_economics)

  service_type n_orders gross_revenue direct_cost gross_margin failure_pct
1     Business      126      12279.23     1655.91     10623.32       19.84
2      Medical      108       9344.88     1379.48      7965.40       14.81
3    Passenger      262      25463.36     3248.56     22214.80       14.50
4       Retail      224      19444.86     2906.27     16538.59       12.50
5       Parcel      230      20735.44     3009.01     17726.43       10.87
    risk_tier
1   High Risk
2   High Risk
3   High Risk
4 Medium Risk
5 Medium Risk


## 10. Summary

Three findings from the SQL diagnostic:

1. **Geographic concentration**: Central is the only zone above 25% failure.
2. **Process control gap**: 26.7% of deliveries use InRepair vehicles at 30.3% failure.
3. **Customer concentration**: 1.5% of customers absorb 12.9% of compensation.

Continue to Notebook 02 for statistical validation.